Imports

In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import numpy as np
from sklearn.metrics import accuracy_score

Load model and reevaluate on clean and master dataset

In [79]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

In [ ]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

FGSM Attack and Evaluate Functions

In [2]:
loss_fn = tf.keras.losses.BinaryCrossentropy()

def fgsm_attack(model, images, labels, epsilon=0.01):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, (-1, 1))  # match (batch,1)

    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images, training=False)
        loss = loss_fn(labels, predictions)

    gradients = tape.gradient(loss, images)
    adv_images = images + epsilon * tf.sign(gradients)
    adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)

    return adv_images

In [3]:
def evaluate_clean(model, generator):
    all_labels = []
    all_preds = []
    all_probs = []

    generator.reset()

    for i in range(len(generator)):
        images, labels = generator[i]

        probs = model.predict(images, verbose=0).flatten()
        preds = (probs > 0.5).astype(int)

        all_labels.extend(labels.astype(int))
        all_probs.extend(probs)
        all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)

    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

In [4]:
def evaluate_fgsm(model, generator, epsilon=0.01):
    all_labels = []
    all_preds = []
    all_probs = []

    generator.reset()

    for i in range(len(generator)):
        images, labels = generator[i]

        images_tf = tf.convert_to_tensor(images, dtype=tf.float32)
        labels_tf = tf.convert_to_tensor(labels, dtype=tf.float32)

        adv_images = fgsm_attack(model, images_tf, labels_tf, epsilon=epsilon)

        probs = model.predict(adv_images, verbose=0).flatten()
        preds = (probs > 0.5).astype(int)

        all_labels.extend(labels.astype(int))
        all_probs.extend(probs)
        all_preds.extend(preds)

    acc = accuracy_score(all_labels, all_preds)

    return np.array(all_labels), np.array(all_preds), np.array(all_probs), acc

Evaluate on their clean dataset

In [83]:
IMG_SIZE = 224
BATCH = 16

datagen = ImageDataGenerator(rescale=1/255.)

Clean dataset (theirs)

In [84]:
ds_clean = datagen.flow_from_directory(
    r"chest_xray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Clean classes:", ds_clean.class_indices)

Found 624 images belonging to 2 classes.
Clean classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Master Dataset

In [85]:
ds_master = datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Master classes:", ds_master.class_indices)

Found 1757 images belonging to 2 classes.
Master classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Kermany Dataset

In [86]:
ds_kermany = datagen.flow_from_directory(
    r"Kermany_Pediatric_Attack",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

print("Kermany classes:", ds_master.class_indices)

Found 468 images belonging to 2 classes.
Kermany classes: {'NORMAL': 0, 'PNEUMONIA': 1}


Evaluate both theirs and master

In [89]:
labels_clean, preds_clean, _, clean_acc = evaluate_clean(model, ds_clean)
labels_master, preds_master, _, master_acc = evaluate_clean(model, ds_master)
labels_master, preds_master, _, kermany_acc = evaluate_clean(model, ds_kermany)


print("Clean (chest_xray):", clean_acc)
print("Clean (master):", master_acc)
print("Clean (kermany):", kermany_acc)

Clean (chest_xray): 0.9214743589743589
Clean (master): 0.6624928856004553
Clean (kermany): 0.8547008547008547


Evaluate on fgsm on both

In [88]:
eps = 0.01

_, _, _, fgsm_clean_acc = evaluate_fgsm(model, ds_clean, epsilon=eps)
_, _, _, fgsm_master_acc = evaluate_fgsm(model, ds_master, epsilon=eps)
_, _, _, fgsm_kermany_acc = evaluate_fgsm(model, ds_kermany, epsilon=eps)

print("FGSM (chest_xray):", fgsm_clean_acc)
print("FGSM (master):", fgsm_master_acc)
print("FGSM (master):", fgsm_kermany_acc)

FGSM (chest_xray): 0.1987179487179487
FGSM (master): 0.12407512805919181
FGSM (master): 0.14957264957264957


Adversial Training

In [47]:
def adversarial_train_step(model, optimizer, images, labels, epsilon=0.01):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, (-1, 1))

    adv_images = fgsm_attack(model, images, labels, epsilon=epsilon)

    combined_images = tf.concat([images, adv_images], axis=0)
    combined_labels = tf.concat([labels, labels], axis=0)

    with tf.GradientTape() as tape:
        predictions = model(combined_images, training=True)
        loss = loss_fn(combined_labels, predictions)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [ ]:
def adversarial_train(model, train_generator, val_generator, epochs=5, epsilon=0.01, lr=1e-4):
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        train_generator.reset()

        train_losses = []

        for i in range(len(train_generator)):
            images, labels = train_generator[i]
            loss = adversarial_train_step(model, optimizer, images, labels, epsilon=epsilon)
            train_losses.append(float(loss))

        print(f"Train loss: {np.mean(train_losses):.4f}")

        _, _, _, clean_acc = evaluate_clean(model, val_generator)
        _, _, _, fgsm_acc = evaluate_fgsm(model, val_generator, epsilon=epsilon)

        print(f"Val clean acc: {clean_acc:.4f}")
        print(f"Val FGSM acc:  {fgsm_acc:.4f}")

Datasets for adv

In [ ]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_train = train_val_datagen.flow_from_directory(
    r"chest_xray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_val = train_val_datagen.flow_from_directory(
    r"chest_xray/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_test = test_datagen.flow_from_directory(
    r"chest_xray/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [54]:
model = tf.keras.models.load_model("baseline_resnet152v2.keras")

adversarial_train(
    model,
    ds_train,
    ds_val,
    epochs=3,
    epsilon=0.01,
    lr=1e-5
)


Epoch 1/3
Train loss: 0.5288
Val clean acc: 0.9608
Val FGSM acc:  0.6794

Epoch 2/3
Train loss: 0.3900
Val clean acc: 0.9636
Val FGSM acc:  0.6947

Epoch 3/3
Train loss: 0.3397
Val clean acc: 0.9675
Val FGSM acc:  0.6976


In [55]:
model.save("baseline_resnet152v2_adversarial.keras")

In [73]:
loss, acc = model.evaluate(ds_test, verbose=1)
print("Test Clean Loss:", loss)
print("Test Clean Accuracy:", acc)

39/39 ━━━━━━━━━━━━━━━━━━━━ 26s 529ms/step - binary_accuracy: 0.8510 - loss: 0.2997
Test Clean Loss: 0.29966625571250916
Test Clean Accuracy: 0.8509615659713745


In [75]:
labels, preds, probs, fgsm_acc = evaluate_fgsm(model, ds_test)

print("Test FGSM Accuracy:", fgsm_acc)

Test FGSM Accuracy: 0.5865384615384616


==================================================================

Adversaraial Training on Master

In [5]:
model = tf.keras.models.load_model("baseline_resnet152v2_adversarial.keras")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 7, 7, 2048)     │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,056,325 (259.61 MB)

 Trainable params: 4,731,137 (18.05 MB)

 Non-trainable params: 53,862,912 (205.47 MB)

 Optimizer params: 9,462,276 (36.10 MB)

Dataset for Master

In [6]:
IMG_SIZE = 224
BATCH = 16
SEED = 42

train_val_datagen = ImageDataGenerator(
    rescale=1/255.,
    validation_split=0.2
)

ds_master_train = train_val_datagen.flow_from_directory(
    r"Master_Dataset/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=True,
    subset="training",
    seed=SEED
)

ds_master_val = train_val_datagen.flow_from_directory(
    r"Master_Dataset/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False,
    subset="validation",
    seed=SEED
)

test_datagen = ImageDataGenerator(rescale=1/255.)

ds_master_test = test_datagen.flow_from_directory(
    r"Master_Dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode="binary",
    batch_size=BATCH,
    shuffle=False
)

Found 11244 images belonging to 2 classes.
Found 350 images belonging to 2 classes.
Found 1757 images belonging to 2 classes.


Regular Training Functions

In [11]:
def train_step(model, optimizer, images, labels):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.float32)
    labels = tf.reshape(labels, (-1, 1))

    with tf.GradientTape() as tape:
        predictions = model(images, training=True)
        loss = loss_fn(labels, predictions)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss


def train(model, train_generator, val_generator, epochs=10, lr=1e-4, patience=2):
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    best_val_loss = float("inf")
    best_weights = None
    wait = 0

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        train_generator.reset()

        train_losses = []

        for i in range(len(train_generator)):
            images, labels = train_generator[i]
            loss = train_step(model, optimizer, images, labels)
            train_losses.append(float(loss))

        print(f"Train loss: {np.mean(train_losses):.4f}")

        val_loss, _, _, val_acc = evaluate_clean(model, val_generator)
        print(f"Val loss: {val_loss:.4f}")
        print(f"Val acc:  {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = model.get_weights()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    if best_weights is not None:
        model.set_weights(best_weights)

    return model

Training

In [ ]:
train(
    model,
    ds_master_train,
    ds_master_val,
    epochs=15,
    lr=1e-5
)


Epoch 1/15


In [ ]:
loss, acc = model.evaluate(ds_master_test, verbose=1)
print("Test Loss:", loss)
print("Test Accuracy:", acc)

In [ ]:
model.save("baseline_resnet152v2_adversarial_finetunedonmaster.keras")